<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z331_Submit_RegLineal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Submit — Regresión Lineal Reciente

Ajusta `tn = a + b·t` sobre los últimos `ventana` meses y extrapola a t+2 (202002).

Cambiá `ventana` en PARAM para probar 3, 6 o 12 meses.

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

descargar  "sell-in.txt.gz"
descargar  "product_id_apredecir201912.txt"

In [ ]:
!pip install uv
!uv pip install -q kaggle

In [ ]:
import os
import numpy as np
import polars as pl
from sklearn.linear_model import LinearRegression

PARAM = {
  'kaggle_competition': 'labo-iii-2026-rosario',
  'ventana': 6,   # meses a usar para la recta: 3, 6 o 12
}

In [ ]:
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
).sort(["product_id", "periodo"])

tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")
tb_ventas    = tb_ventas.join(tb_apredecir, on="product_id", how="inner").sort(["product_id", "periodo"])

productos = tb_apredecir["product_id"].to_list()
print(f"{len(productos)} productos")

In [ ]:
resultados = []
ventana = PARAM['ventana']

for pid in productos:
    serie = (
        tb_ventas.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )

    w = min(ventana, len(serie))
    y = serie[-w:]
    x = np.arange(w).reshape(-1, 1)

    if w >= 2:
        m    = LinearRegression().fit(x, y)
        pred = float(m.predict([[w + 1]])[0])  # t+2
    else:
        pred = float(serie.mean())

    resultados.append({'product_id': pid, 'tn': max(pred, 0.0)})

tb_final = pl.DataFrame(resultados)
display(tb_final.head(10))
print(f"Nulls: {tb_final['tn'].is_null().sum()}")

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
    comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
    os.system(comando)

archivo = f"RegLineal_{ventana}m.csv"
tb_final.write_csv(archivo)
kaggle_submit(PARAM['kaggle_competition'], archivo, f'Regresion lineal ultimos {ventana} meses t+2')